In [1]:
import pandas as pd
import os

complaints_folder = "/Users/davishunter/Desktop/complaints/"
complaints_2020_file = os.path.join(complaints_folder, "2020_complaints.csv")
complaints_2021_file = os.path.join(complaints_folder, "2021_complaints.csv")
complaints_2022_file = os.path.join(complaints_folder, "2022_complaints.csv")
complaints_2023_file = os.path.join(complaints_folder, "2023_complaints.csv")

In [2]:
df_2020 = pd.read_csv(complaints_2020_file)
df_2021 = pd.read_csv(complaints_2021_file)
df_2022 = pd.read_csv(complaints_2022_file)
df_2023 = pd.read_csv(complaints_2023_file)

df_all_years = pd.concat([df_2020, df_2021, df_2022, df_2023], ignore_index=True)
print(df_all_years.columns)

Index(['model_year', 'make', 'model', 'products_type', 'components',
       'complaint_text'],
      dtype='object')


In [3]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import os
import numpy as np


# Remove rows where complaint_text is missing (NaN) or empty
df_all_years = df_all_years.dropna(subset=['complaint_text'])

# Ensure all text is treated as string (fixes random non-string numbers)
df_all_years['complaint_text'] = df_all_years['complaint_text'].astype(str)

# Reset index so your unique_id remains clean and sequential
df_all_years = df_all_years.reset_index(drop=True)
df_all_years['unique_id'] = df_all_years.index


model_name = 'all-MiniLM-L6-v2'
model = SentenceTransformer(model_name)

# B. Extract the text
texts_to_embed = df_all_years['complaint_text'].tolist()

embeddings = model.encode(texts_to_embed, show_progress_bar=True, convert_to_numpy=True)

# D. Create the final mapping DataFrame
# This DataFrame links the unique ID to the corresponding embedding vector.
embeddings_df = pd.DataFrame({
    'unique_id': df_all_years['unique_id'],
    'embedding': list(embeddings) # Store the numpy array as a list/array object in the cell
})


/opt/anaconda3/envs/p39/lib/python3.9/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Batches:   0%|          | 0/7338 [00:00<?, ?it/s]

In [4]:
df_final = pd.merge(df_all_years, embeddings_df, on='unique_id', how='inner')

parquet_filename = "complaints_with_embeddings.parquet"
df_final.to_parquet(parquet_filename, engine='pyarrow', index=False)

print(f"✅ Successfully saved {len(df_final)} rows to {parquet_filename}")

✅ Successfully saved 234814 rows to complaints_with_embeddings.parquet


In [5]:
df_loaded = pd.read_parquet(parquet_filename)

print("\nLoaded Data Preview:")
print(df_loaded[['unique_id', 'complaint_text', 'embedding']].head(2))

# Verify the embedding is still a correct list/array
print(f"\nEmbedding type check: {type(df_loaded['embedding'].iloc[0])}")


Loaded Data Preview:
   unique_id                                     complaint_text  \
0          0  [XXX]  [XXX]  Email: [XXX]   Subject: CarMax a...   
1          1  I live in a colder climate, we’re were experie...   

                                           embedding  
0  [-0.07601695, 0.0838188, 0.05429085, -0.029568...  
1  [-0.08385411, -0.017931452, 0.067818634, 0.100...  

Embedding type check: <class 'numpy.ndarray'>


In [8]:
import numpy as np
import umap
import plotly.express as px
from sklearn.cluster import KMeans

# --- STEP 0: Create the 10k Sample ---
print("Sampling 10,000 rows for speed...")

# Get random indices
# (If you have fewer than 10k rows, we just take them all)
sample_size = min(10000, len(df_all_years))
indices = np.random.choice(len(df_all_years), sample_size, replace=False)

# Create the subsets
# .iloc[] selects rows by index position
df_subset = df_all_years.iloc[indices].copy()
embeddings_subset = embeddings[indices]

print(f"Working with {len(df_subset)} rows.")


# --- STEP 1: Clustering (on the subset) ---
print("Step 1: Clustering data...")
kmeans = KMeans(n_clusters=15, random_state=42)
clusters = kmeans.fit_predict(embeddings_subset)
df_subset['cluster_label'] = clusters.astype(str)


# --- STEP 2: UMAP (on the subset) ---
print("Step 2: Running UMAP (this will now be very fast)...")
reducer = umap.UMAP(n_neighbors=15, n_components=2, metric='cosine', random_state=42)
umap_2d = reducer.fit_transform(embeddings_subset)

# Add coordinates to the SUBSET dataframe
df_subset['x_coord'] = umap_2d[:, 0]
df_subset['y_coord'] = umap_2d[:, 1]


# --- STEP 3: Plotting ---
print("Step 3: Generating plot...")
df_subset['short_text'] = df_subset['complaint_text'].str.slice(0, 50) + "..."

fig = px.scatter(
    df_subset,
    x='x_coord',
    y='y_coord',
    color='cluster_label',
    hover_data=['short_text', 'unique_id'],
    title='Galaxy of Complaints (10k Sample)',
    template='plotly_dark',
    width=1000,
    height=800
)

fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)

fig.show()

Sampling 10,000 rows for speed...
Working with 10000 rows.
Step 1: Clustering data...
Step 2: Running UMAP (this will now be very fast)...


/opt/anaconda3/envs/p39/lib/python3.9/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



Step 3: Generating plot...
